# Two legacy physics mistakes on P1 walking: what each one costs per joint, with nothing else changed

This notebook reports on the following key findings about our reanalysis:

1. **Only the equations differ.** Same subject (P1, trial 13, walking at 1.7 m/s), same markers, anthropometrics, joint centres, forces, filtering and 12-segment chain. The joint *forces* come out identical across all variants, so nothing else drifted.
2. **What the two real legacy mistakes cost.** The one-sided inertia rotation (a) plus the missing gyroscopic term (b) move the joint torque by 0.46 / 3.37 / 8.38 N m RMS at the right ankle / knee / hip (0.26 / 2.21 / 6.65% of peak), and 0.54 / 2.75 / 8.23 N m on the left. The ankle sits below the ~1.5–2 N m we agree with Visual3D; the hip sits well above it.
3. **The one-sided inertia is nearly all of it.** At the right hip, (a) alone costs 8.17 N m RMS and (b) alone 0.47 N m.
4. **Why walking hides the gyroscopic term.** The thigh spends 79% of its |ω| on its medio-lateral (principal) axis, so ω × (Iω) peaks at 1.62 N m over the whole right leg, against an Iα term of 15.49 N m at the thigh and a 126 N m peak hip torque.
5. **The energy audit catches it with no reference.** The imbalance is 1.87 W RMS with correct torques and 16.10 W RMS with (a)+(b). No Visual3D, no ground truth needed.

What this page is *not*: the full legacy pipeline. The third real error (RoomSegAng's `TMdot*TM` angular velocity) and the free moment in the wrong slot aren't switched on here. Those, plus the executed MATLAB itself, are in [asics_running_c3d](asics_running_c3d.ipynb) steps 14–18. The derivations of each mistake are in [legacy_error_math](legacy_error_math.ipynb). Some tables and curves below also carry a "(c)" variant, a *hypothetical* Cardan-angle acceleration. It is not what the legacy code did (I got that wrong at first); treat it as a sensitivity check only.

In [1]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "boneid @ git+https://github.com/jeremydwong/bonelab_inverse_dynamics"],
        check=True,
    )

# Works from the repository root, notebooks/, or an exported script.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").exists()
             and (p / "src/boneid").is_dir()), Path.cwd())
SAMPLE_PATH = ROOT / "data/p1_trial13_sample.mat"
if IN_COLAB and not SAMPLE_PATH.exists():
    from urllib.request import urlretrieve
    SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(
        "https://raw.githubusercontent.com/jeremydwong/bonelab_inverse_dynamics/"
        "main/data/p1_trial13_sample.mat", SAMPLE_PATH,
    )

## 1 — Same inputs, only the equations differ

The bundled file holds original trial 13 at slot 0. For the full P1 export, change `MAT_PATH` and set `TRIAL_INDEX = 13`.

`asics_comparison` runs the whole-body recursion several times with a flags record set differently: `correct`, `ab` (the two verified mistakes), `abc` (plus the hypothetical Cardan term), and each flag alone. The assert below is the "nothing else drifted" check: F = m a − F<sub>d</sub> − m g has no inertia tensor in it, so the joint forces have to match bit for bit.

In [2]:
import numpy as np
from IPython.display import HTML, display
from boneid import io_v3d as io
from boneid import report as R

MAT_PATH = SAMPLE_PATH
TRIAL_INDEX = 0  # bundled sample; use 13 with the full P1 export
params = R.p1_params()
trial = io.load_v3d_trial(MAT_PATH, TRIAL_INDEX)
# Preserve the original trial label in the mirrored report.
if MAT_PATH.resolve() == SAMPLE_PATH.resolve():
    trial.index = 13
print(f"P1 trial {trial.index}: {len(trial.t)} frames at {trial.rate:g} Hz")
print(params)

P1 trial 13: 637 frames at 120 Hz
AnalysisParams(gravity=array([ 0.  ,  0.  , -9.81]), contact_threshold_n=20.0, lowpass_hz=12.0, force_lowpass_hz=0.0, filter_order=4, treadmill_speed=0.0, flag_crossover=False)


In [3]:
data = R.asics_comparison(trial, params)
for variant in ("ab", "abc"):
    for side in ("right", "left"):
        np.testing.assert_array_equal(
            getattr(data["runs"][variant], side).joint_force,
            getattr(data["runs"]["correct"], side).joint_force,
        )
print("Joint forces are identical across all three variants.")
for variant in R.ASICS_VARIANTS:
    print(f"{variant}: energy imbalance "
          f"{data['imbalance'][variant]['rms']:.2f} W RMS")

Joint forces are identical across all three variants.
correct: energy imbalance 1.87 W RMS
ab: energy imbalance 16.10 W RMS
abc: energy imbalance 17.21 W RMS


The report HTML is built once and the cells below show its sections one at a time (the figures keep their shared left/right axes).

In [4]:
import re

report_html = R.p1_asics_report_html(trial, data, params)
body = re.search(r"<main>(.*?)</main>", report_html, re.S).group(1)
parts = re.split(r"(<h2\b[^>]*>.*?</h2>)", body, flags=re.S)
sections = {"Overview": parts[0]}
for i in range(1, len(parts), 2):
    title = re.sub(r"<[^>]+>", "", parts[i])
    sections[title] = parts[i + 1]

NOTEBOOK_STYLE = """<style>
.asics-notebook {font:16px/1.55 system-ui,sans-serif; color:#1c1b18;
 background:#f7f6f2; padding:20px; max-width:100%;}
.asics-notebook img {max-width:100%; height:auto;}
.asics-notebook figure {margin:16px 0; padding:12px; background:#fcfcfb;}
.asics-notebook figcaption {font-size:0.85em; color:#52514e;}
.asics-notebook .overflow {overflow-x:auto;}
.asics-notebook table {border-collapse:collapse; font-size:0.9em;}
.asics-notebook td, .asics-notebook th {padding:6px 12px;
 border-bottom:1px solid #dedcd4; text-align:right;}
.asics-notebook td:first-child, .asics-notebook th:first-child {text-align:left;}
.asics-notebook .missing-files td, .asics-notebook .missing-files th {
 text-align:left; vertical-align:top;}
.asics-notebook .stat-row {display:flex; flex-wrap:wrap; gap:12px; margin:16px 0;}
.asics-notebook .stat {background:#fcfcfb; padding:12px; flex:1; min-width:160px;}
.asics-notebook .k {font-size:0.8em; color:#52514e;}
.asics-notebook .v {font-size:1.4em; font-weight:600;}
.asics-notebook .u {font-size:0.7em; font-weight:400;}
.asics-notebook .pass {color:#0a7d43;} .asics-notebook .fail {color:#b3261e;}
</style>"""

def show_section(title):
    content = re.sub(r"<style>.*?</style>", "", sections[title], flags=re.S)
    display(HTML(NOTEBOOK_STYLE + '<div class="asics-notebook">'
                 + content + '</div>'))

## 2 — What the two mistakes cost, joint by joint

RMS of the full 3-vector torque difference per joint. The (a)+(b) columns are the real mistakes. Figures 1–3 overlay the torques and powers; at whole-trial scale they're one curve, and only the zoomed stride shows the hip and knee separating in swing.

In [5]:
show_section('What it costs, joint by joint')

joint,peak |τ| (N m),(a)+(b) RMS Δτ (N m),% of peak,(a)+(b)+(c) RMS Δτ (N m),% of peak,peak |Δτ| (N m)
R ankle,173,0.46,0.26,0.40,0.23,1.40
R knee,153,3.37,2.21,1.85,1.22,6.69
R hip,126,8.38,6.65,6.82,5.41,17.30
L ankle,157,0.54,0.35,0.69,0.44,2.34
L knee,155,2.75,1.78,3.16,2.04,11.98
L hip,149,8.23,5.54,8.69,5.85,22.39


## 3 — Which mistake dominates: the one-sided inertia

Same call, one flag at a time. The single-flag table is the one to read. The term-magnitude table under it says why (b) barely matters here. (Note (a) and the hypothetical (c) partially cancel; that's luck, not structure.)

In [6]:
show_section('Which mistake dominates')

## 4 — Why walking hides it

The gyroscopic part is easy: ω rides a principal axis. For the one-sided inertia it's different. Planar motion does *not* forgive it in general. R·I·Rᵀ·α equals R·I·α only when α lies along R's own rotation axis, i.e. the segment frame started out aligned with the lab and has turned about one fixed axis since. Real segments carry a static offset from the lab plus some out-of-plane wobble, so the error is really there (8.17 N m at the right hip). It's just a mistake inside the inertial term, which is a small part of the balance in walking. Heads up: the per-component table in this section uses the (a)+(b)+(c) variant.

In [7]:
show_section('Honest about the magnitude')

joint,Δτx,Δτy,Δτz,% peak x,% peak y,% peak z
R ankle,0.20,0.27,0.23,0.1,1.0,0.9
R knee,1.00,1.30,0.88,0.8,1.4,3.0
R hip,4.05,4.26,3.47,3.3,4.3,9.2
L ankle,0.59,0.25,0.26,0.4,1.3,1.0
L knee,2.70,1.30,1.00,1.9,1.7,8.4
L hip,6.12,5.08,3.50,4.5,4.5,13.8


## 5 — The energy audit catches it with no reference

This is the reason I built the audit. It asks only whether the model's own dE/dt equals the power flowing in through joints, ground and residual.

In [8]:
show_section('The audit catches it without a reference')

## Where this goes next

Previous: [legacy_error_math](legacy_error_math.ipynb) (the math of each mistake). The full legacy port, the correction ladder and the executed MATLAB on real running data: [asics_running_c3d](asics_running_c3d.ipynb). Back to the [index](../reports/toc.html).